In [1]:
import pandas as pd
import networkx as nx
import numpy as np
from pathlib import Path

def process_fastspar_group(group: str, base_dir: str = ".") -> pd.DataFrame:    
    base_path = Path(base_dir) / group
    corr_file = base_path / f"{group}_median_correlation.tsv"
    pval_file = base_path / f"{group}_pvalues.tsv"
    
    # 데이터 로드
    corr_df = pd.read_csv(corr_file, sep='\t', index_col=0)
    pval_df = pd.read_csv(pval_file, sep='\t', index_col=0)

    pval_threshold = 0.001
    
    # Edge list 생성
    results = []
    taxa_list = corr_df.index.tolist()
    
    for i, source in enumerate(taxa_list):
        for j, target in enumerate(taxa_list):
            if i < j:  
                correlation = corr_df.loc[source, target]
                p_value = pval_df.loc[source, target]
                if pd.notna(correlation) and p_value <= pval_threshold:
                    results.append({
                        'source': source,
                        'target': target,
                        'correlation': correlation,
                        'p_value': p_value
                    })
    
    return pd.DataFrame(results)

def process_all_groups(groups: list = None, base_dir: str = "."):
    
    if groups is None:
        groups = ['all', 'c0', 'c1', 'c2', 'c3']
    results = {}

    for group in groups:
        result = process_fastspar_group(group, base_dir)
        results[group] = result
    return results

result_dict = process_all_groups(base_dir="data/3-results/fastspar")

In [2]:
def load_network_graphs(groups: list = None, base_dir: str = ".",
                        corr_threshold: float = 0.3, pval_threshold: float = 0.001, prevalence_threshold: float = 0.4) -> dict:
    if groups is None:
        groups = ['all', 'c0', 'c1', 'c2', 'c3']
    all_graphs = {}
    
    for group in groups:
        df = result_dict[group]
        df = df[(df['p_value'] <= pval_threshold)
                 & (df['correlation'].abs() >= corr_threshold)
                   ]

        G = nx.from_pandas_edgelist(
            df,
            source='source',
            target='target',
            edge_attr=['correlation', 'p_value',
                        ])
        
        all_graphs[group] = G
    
    return all_graphs

if __name__ == "__main__":
    all_graphs = load_network_graphs(corr_threshold=0.3, base_dir='data/3-results/fastspar', pval_threshold=0.001, prevalence_threshold=0.4)
    
    for name, G in all_graphs.items():
        print(f"{name}: {nx.number_of_nodes(G)} nodes, {nx.number_of_edges(G)} edges")
    

all: 98 nodes, 285 edges
c0: 69 nodes, 135 edges
c1: 90 nodes, 191 edges
c2: 66 nodes, 113 edges
c3: 98 nodes, 242 edges


In [4]:
groups = ['all', 'c0', 'c1', 'c2', 'c3']
for group in groups:
    base_path = Path('data/3-results/fastspar') / group
    edge_file = base_path / f"{group}_network_edges.csv"
    df = pd.read_csv(edge_file)
for name, G in all_graphs.items():
    print(f"{name}: {nx.number_of_nodes(G)} nodes, {nx.number_of_edges(G)} edges")

all: 98 nodes, 285 edges
c0: 69 nodes, 135 edges
c1: 90 nodes, 191 edges
c2: 66 nodes, 113 edges
c3: 98 nodes, 242 edges


In [5]:
def calculate_centrality_and_save_to_folders(all_graphs: dict, 
                                             weight_attr: str = "correlation", 
                                             base_dir: str = ".") -> dict:    
    centrality_results = {}
    
    for name, G in all_graphs.items():
        bc = nx.betweenness_centrality(G, weight=weight_attr, normalized=False)
        dc = nx.degree_centrality(G)
        bc_df = pd.DataFrame(bc.items(), columns=["Node", "Betweenness"])
        bc_df = bc_df.sort_values("Betweenness", ascending=False)
        dc_df = pd.DataFrame(dc.items(), columns=["Node", "DegreeCentrality"])  
        dc_df = dc_df.sort_values("DegreeCentrality", ascending=False)
        
        centrality_results[name] = {
            'betweenness': bc_df,
            'degree': dc_df
            }
        
        print(f"\n{name} degree centrality (Top 5)")
        print(dc_df.head())
        print(f"\n{name} betweenness centrality (Top 5)")
        print(bc_df.head())

        # 각 그룹 폴더에 저장
        group_folder = Path(base_dir) / name
        dc_file = group_folder / f"{name}_degree_centrality.tsv"
        bc_file = group_folder / f"{name}_betweenness_centrality.tsv"
        
        dc_df.to_csv(dc_file, sep='\t', index=False)
        bc_df.to_csv(bc_file, sep='\t', index=False)
    
    return centrality_results

centrality_results = calculate_centrality_and_save_to_folders(all_graphs, base_dir='data/3-results/fastspar')



all degree centrality (Top 5)
                 Node  DegreeCentrality
3   g__Ruminococcus_B          0.340206
15        g__Gemmiger          0.268041
11  g__Thomasclavelia          0.257732
23     g__Veillonella          0.226804
29    g__Faecalimonas          0.226804

all betweenness centrality (Top 5)
                   Node  Betweenness
3     g__Ruminococcus_B       3301.5
15          g__Gemmiger       3290.0
29      g__Faecalimonas       3149.0
9   g__Faecalibacterium       3086.0
11    g__Thomasclavelia       2987.0

c0 degree centrality (Top 5)
                   Node  DegreeCentrality
12  g__Faecalibacterium          0.191176
39     g__Streptococcus          0.132353
28           g__Dorea_A          0.132353
15       g__Veillonella          0.132353
21       g__Bacteroides          0.117647

c0 betweenness centrality (Top 5)
                     Node  Betweenness
12    g__Faecalibacterium       1763.0
15         g__Veillonella       1762.5
52  g__Lacticaseibacillus       1499.

In [7]:
def calculate_centrality_df(all_graphs: dict, 
                            weight_attr: str = "correlation", 
                            base_dir: str = ".") -> pd.DataFrame:
    centrality_results = []
    
    for name, G in all_graphs.items():
        # Betweenness
        bc = nx.betweenness_centrality(G, weight=weight_attr, normalized=False)
        bc_df = pd.DataFrame(bc.items(), columns=["Node", "Betweenness"])

        # Degree
        dc = nx.degree_centrality(G)
        dc_df = pd.DataFrame(dc.items(), columns=["Node", "DegreeCentrality"])

        # wide-format으로 합치기
        cluster_df = pd.merge(dc_df, bc_df, on="Node")
        cluster_df["Cluster"] = name

        centrality_results.append(cluster_df)

        # 결과 출력
        print(f"\n{name} degree centrality (Top 5)")
        print(dc_df.sort_values("DegreeCentrality", ascending=False).head())
        print(f"\n{name} betweenness centrality (Top 5)")
        print(bc_df.sort_values("Betweenness", ascending=False).head())

        # 각 그룹 폴더에 저장
        group_folder = Path(base_dir) / name
        group_folder.mkdir(parents=True, exist_ok=True)
        cluster_df.to_csv(group_folder / f"{name}_centrality_wide_betweenness.tsv", sep="\t", index=False)

    # 전체 결과 합치기
    all_results = pd.concat(centrality_results, ignore_index=True)
    return all_results
results = calculate_centrality_df(all_graphs, base_dir='data/3-results/fastspar')




all degree centrality (Top 5)
                 Node  DegreeCentrality
3   g__Ruminococcus_B          0.340206
15        g__Gemmiger          0.268041
11  g__Thomasclavelia          0.257732
23     g__Veillonella          0.226804
29    g__Faecalimonas          0.226804

all betweenness centrality (Top 5)
                   Node  Betweenness
3     g__Ruminococcus_B       3301.5
15          g__Gemmiger       3290.0
29      g__Faecalimonas       3149.0
9   g__Faecalibacterium       3086.0
11    g__Thomasclavelia       2987.0

c0 degree centrality (Top 5)
                   Node  DegreeCentrality
12  g__Faecalibacterium          0.191176
39     g__Streptococcus          0.132353
28           g__Dorea_A          0.132353
15       g__Veillonella          0.132353
21       g__Bacteroides          0.117647

c0 betweenness centrality (Top 5)
                     Node  Betweenness
12    g__Faecalibacterium       1763.0
15         g__Veillonella       1762.5
52  g__Lacticaseibacillus       1499.